# Fast ℓp Proximity Operator — Phase 1
## Accuracy-matched GPU benchmarking & reproducible data archive

**Purpose.** This is a clean replacement for the Phase 1 exploratory Colab, reconstructed from the recorded CSVs and the public benchmark CLI. The original interactive notebook was not provided; the cells below are organized as a fresh, reproducible workflow rather than a literal revision of that file.

**What is inside the companion ZIP?** Six **formal** A100 runs (1,000,000 inputs, float32, 30 timing repeats), two **pilot** runs (10,000 inputs), three **smoke** runs, derived comparison tables, QA/integrity metadata, figures, and a deterministic report-building script.

**Key safeguards**
- Never mix formal/pilot/smoke timing data.
- Raw CSVs remain unchanged; derived outputs are reproducibly regenerated.
- Online runtime excludes Chebyshev offline fitting/reference construction.
- The normalized error criterion is **measured** on the recorded sample; it is **not** a whole-domain certificate.
- If you rerun, results go under `data/rerun/` and do **not** overwrite archived runs.
- No reviewer correspondence, internal response draft, or manuscript editing is included.

**Suggested order:** Run sections A–D for analysis without a GPU. Use section E only if you want to rerun one or more formal experiments. Finish with section F to export a new ZIP.

---
## A · Load archived experimental data

Open this `.ipynb` in Google Colab. When prompted, **upload `FastLp_Phase1_Experiment_Package.zip`** (the accompanying file). No GPU is needed to read existing data.

If running locally inside the extracted package, the notebook finds the `data/formal/` directory automatically.

In [ ]:
from pathlib import Path
import os, sys, json, zipfile, subprocess, shutil
import pandas as pd
from IPython.display import display, Image

ROOT = None
for candidate in [Path.cwd(), Path.cwd().parent, Path('/content/fast_lp_phase1_package')]:
    if (candidate / 'data' / 'formal').is_dir():
        ROOT = candidate.resolve()
        break

if ROOT is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Extract the companion ZIP next to this notebook, or use Colab upload.') from exc
    print('Upload FastLp_Phase1_Experiment_Package.zip')
    uploaded = files.upload()
    candidates = [Path(name) for name in uploaded if name.lower().endswith('.zip')]
    if len(candidates) != 1:
        raise ValueError('Please upload exactly one Phase 1 ZIP archive.')
    ROOT = Path('/content/fast_lp_phase1_package')
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(candidates[0]) as zf:
        for member in zf.namelist():
            p = Path(member)
            if p.is_absolute() or '..' in p.parts:
                raise ValueError('Unsafe archive path: ' + member)
        zf.extractall(ROOT)

FORMAL_DIR = ROOT / 'data' / 'formal'
assert FORMAL_DIR.is_dir(), f'Formal CSVs not found at {FORMAL_DIR}'
print('Bundle root:', ROOT)
print('Formal CSVs:', len(list(FORMAL_DIR.glob('*.csv'))))
print('Pilot CSVs:', len(list((ROOT/'data'/'pilot').glob('*.csv'))))
print('Smoke CSVs:', len(list((ROOT/'data'/'smoke').glob('*.csv'))))


## B · Rebuild tables, figures, and quality checks from raw CSVs

This step invokes the included analysis script. It **does not** run a GPU benchmark and **does not** change any raw CSV.

In [ ]:
analysis_script = ROOT / 'scripts' / 'rebuild_phase1_artifacts.py'
subprocess.run([sys.executable, str(analysis_script), '--root', str(ROOT)], check=True)
print((ROOT/'reports'/'quality_summary.json').read_text())


In [ ]:
best = pd.read_csv(ROOT / 'tables' / 'formal_best_comparison.csv')
cols = [
    'p_label', 'chebyshev_configuration', 'chebyshev_median_ms',
    'newton_configuration', 'newton_median_ms',
    'bisection_configuration', 'bisection_median_ms',
    'cheb_over_newton'
]
display(best[cols].round(4))


### Measurement definition

The accuracy filter uses

$$
E_\infty=\max_i\frac{|\widehat x_i-x_i^{\mathrm{ref}}|}{\max\{|y_i|,\tau_p\}}\le 10^{-6},
$$

along with finite outputs and zero support mismatches for the derived summary. `max_active_stationarity_residual` is a **separate, unnormalized** diagnostic; it is not the accuracy filter.

The reference is float64 bracketed bisection (75 iterations), cross-checked against independent float64 Newton on a subset. The current protocol records medians/standard deviations but **not** the individual timing repetitions.

### Data-quality audit

In [ ]:
quality = pd.read_csv(ROOT / 'reports' / 'formal_quality_checks.csv')
display(quality[['p_label','n_configurations','n_pass_accuracy',
                 'n_nonfinite','total_support_mismatches',
                 'max_reference_crosscheck_norm_error','commit']])

all_rows = pd.read_csv(ROOT / 'tables' / 'formal_all_configurations.csv')
assert len(all_rows) == 112
assert (all_rows['n_timing'] == 1_000_000).all()
assert (all_rows['dtype'] == 'float32').all()
assert (all_rows['support_mismatch_count'] == 0).all()
print('Formal rows validated:', len(all_rows))


## C · Figures for the formal runs

Figures are generated directly from the CSVs and saved as both 220-dpi PNG and vector PDF. The stars in the six-panel figure indicate the **fastest passing** configuration within the tested grid.

In [ ]:
for filename in ['best_runtime.png','cheb_newton_time_ratio.png','accuracy_runtime_six_panel.png']:
    print(filename)
    display(Image(filename=str(ROOT / 'figures' / filename)))


## D · Explore an individual exponent and export the tables

Change `P_TO_EXPLORE` to inspect all recorded Chebyshev/Newton/Bisection configurations for one exponent. The selection does not alter the raw data.

In [ ]:
P_TO_EXPLORE = '0.999'  # '1/3', '2/5', '3/4', '0.9', '0.99', or '0.999'
sub = all_rows[all_rows['p_label'].astype(str) == P_TO_EXPLORE].copy()
fields = ['configuration','median_ms','std_ms','max_norm_error',
          'max_active_stationarity_residual','meets_accuracy_target',
          'sampled_float64_branch_error']
display(sub[fields].sort_values('median_ms').reset_index(drop=True))


---
## E · Optional: independently reproduce formal runs on A100 / CUDA

**Do not enable unless you want new experiments.** The original six full runs are already inside the ZIP. Re-running may take substantial Colab time and may yield different timings on a different runtime/GPU.

Recorded formal code revision: `7a1e32b`, public branch `revision/major-experiments`, repository `https://github.com/Jy-U1/Fast_Lp_Prox`. We deliberately pin the commit instead of silently using a moving branch tip.

- To run **one** exponent first, set `RUN_BENCHMARKS=True`, `SELECTED_RUNS=['p0333']`.
- To run all six formal sweeps, set `SELECTED_RUNS` to all six IDs.
- Results are written to `data/rerun/` only; original formal CSVs are preserved.

The source installation step requires internet/GitHub access; GPU execution requires a CUDA runtime.

In [ ]:
RUN_BENCHMARKS = False  # EXPLICITLY opt in
SELECTED_RUNS = ['p0333']  # p0333, p0400, p0750, p0900, p0990, p0999
PINNED_COMMIT = '7a1e32b'
REPO_DIR = Path('/content/Fast_Lp_Prox')

if RUN_BENCHMARKS:
    import torch
    assert torch.cuda.is_available(), 'Select Colab Runtime > Change runtime type > GPU'
    print('GPU:', torch.cuda.get_device_name(0))
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git','clone','--branch','revision/major-experiments',
                        'https://github.com/Jy-U1/Fast_Lp_Prox.git',str(REPO_DIR)],check=True)
    subprocess.run(['git','fetch','origin','revision/major-experiments'], cwd=REPO_DIR,check=True)
    subprocess.run(['git','checkout','--detach',PINNED_COMMIT],cwd=REPO_DIR,check=True)
    commit = subprocess.check_output(['git','rev-parse','--short','HEAD'],cwd=REPO_DIR,text=True).strip()
    assert commit == PINNED_COMMIT, (commit, PINNED_COMMIT)
    subprocess.run([sys.executable,'-m','pip','install','-e','.[dev]'],cwd=REPO_DIR,check=True)
    subprocess.run([sys.executable,'-m','pytest','-q',
                    'tests/test_revision_accuracy_runtime.py'],cwd=REPO_DIR,check=True)
    print('Reproduction source verified:',commit)
else:
    print('RUN_BENCHMARKS=False: using archived CSVs; no GPU work launched.')


In [ ]:
# Build the exact candidate grids from the archived formal CSV metadata.
# This avoids copying degree/iteration lists by hand.
if RUN_BENCHMARKS:
    specs = json.loads((ROOT/'reports'/'formal_parameter_grids.json').read_text())
    specs = {spec['tag']:spec for spec in specs}
    permitted = set(specs)
    if not set(SELECTED_RUNS).issubset(permitted):
        raise ValueError(f'Unknown run ID; choose from {sorted(permitted)}')
    (ROOT/'data'/'rerun').mkdir(exist_ok=True)
    for run_id in SELECTED_RUNS:
        item = specs[run_id]
        info = item['common']
        target = ROOT/'data'/'rerun'/item['source_file'].replace('.csv','_rerun.csv')
        if target.exists():
            print('SKIP existing rerun (no overwrite):', target)
            continue
        argv = [
            sys.executable,'scripts/benchmark_accuracy_runtime.py',
            '--device','cuda','--dtype','float32',
            '--p',item['p_label'],'--lam',str(info['lambda']),
            '--n',str(info['n_timing']),
            '--active-ratio',str(info['active_ratio_requested']),
            '--ymax-factor',str(info['ymax_factor']),
            '--seed',str(info['seed']),
            '--cheb-degrees',','.join(map(str,item['cheb_degrees'])),
            '--newton-iters-grid',','.join(map(str,item['newton_iterations'])),
            '--bisect-iters-grid',','.join(map(str,item['bisection_iterations'])),
            '--n-fit',str(item['n_fit']),
            '--n-val',str(info['validation_n']),
            '--warmup',str(info['warmup']),
            '--repeats',str(info['repeats']),
            '--target-norm-error',str(info['accuracy_target']),
            '--output',str(target),
        ]
        print('Re-running',item['p_label'], 'with',len(item['cheb_degrees']),
              'Chebyshev degrees,',len(item['newton_iterations']),'Newton steps,',
              len(item['bisection_iterations']),'bisection steps')
        subprocess.run(argv,cwd=REPO_DIR,check=True)
    print('New rerun CSV files:',sorted(p.name for p in (ROOT/'data'/'rerun').glob('*.csv')))


## E.2 · Compare any independent reruns with the archived baseline

The rerun timings are shown **separately**. They are not silently merged with the original A100 runs.

In [ ]:
rerun_dir = ROOT/'data'/'rerun'
if rerun_dir.is_dir() and list(rerun_dir.glob('*.csv')):
    comparisons = []
    for new_file in sorted(rerun_dir.glob('*_rerun.csv')):
        original_file = FORMAL_DIR/new_file.name.replace('_rerun.csv','.csv')
        if not original_file.exists():
            continue
        base = pd.read_csv(original_file)
        new = pd.read_csv(new_file)
        join = base[['configuration','median_ms','max_norm_error']].merge(
            new[['configuration','median_ms','max_norm_error']],
            on='configuration',suffixes=('_archived','_rerun'))
        join['source_file'] = original_file.name
        join['rerun_over_archived'] = join['median_ms_rerun']/join['median_ms_archived']
        comparisons.append(join)
    if comparisons:
        display(pd.concat(comparisons,ignore_index=True))
else:
    print('No reruns yet. Archived formal data remain unchanged.')


---
## F · Export a clean ZIP

This packages the archived **raw CSVs**, any new `data/rerun/` CSVs, regenerated tables, plots, QA reports, rebuild script, README, and notebook. It excludes external repository files and private reviewer/manuscript materials.

In [ ]:
EXPORT = Path('/content/FastLp_Phase1_Colab_Export.zip') if Path('/content').exists() else Path.cwd()/'FastLp_Phase1_Colab_Export.zip'
with zipfile.ZipFile(EXPORT,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=8) as out:
    for file in sorted(ROOT.rglob('*')):
        if not file.is_file() or '.ipynb_checkpoints' in file.parts or '__pycache__' in file.parts:
            continue
        out.write(file,file.relative_to(ROOT))
print('Exported:',EXPORT, 'size:',EXPORT.stat().st_size,'bytes')
try:
    from google.colab import files
    files.download(str(EXPORT))
except ImportError:
    print('Local file ready:',EXPORT)


### What Phase 1 does **not** prove

The six speed-vs-accuracy sweeps are for recorded PyTorch scalar proximal kernels on a sampled input distribution. In particular, they do not certify all active-branch inputs, prove numerical robustness directly at the activation threshold, supply an end-to-end ℓp-Lasso convergence experiment, or compare specialized fused GPU implementations.

**Next experimental phase (not included here):** deterministic threshold-neighborhood tests, float32 vs float64 checks, cross-device/input-size validation, and randomized/interleaved benchmarking for close speed comparisons. Those are separate experiments and should have separate artifacts.